# Reordering Supply Stacks

## Background

The expedition can depart as soon as the final supplies have been unloaded from the ships. Supplies are stored in stacks of marked crates, but because the needed supplies are buried under many other crates, the crates need to be rearranged.

The ship has a giant cargo crane capable of moving crates between stacks. To ensure none of the crates get crushed or fall over, the crane operator will rearrange them in a series of carefully-planned steps. After the crates are rearranged, the desired crates will be at the top of each stack.

The Elves don't want to interrupt the crane operator during this delicate procedure, but they forgot to ask her which crate will end up where, and they want to be ready to unload them as soon as possible so they can embark.

They do, however, have a drawing of the starting stacks of crates and the rearrangement procedure (your puzzle input). For example:

```
    [D]    
[N] [C]    
[Z] [M] [P]
 1   2   3

move 1 from 2 to 1
move 3 from 1 to 3
move 2 from 2 to 1
move 1 from 1 to 2
```

In this example, there are three stacks of crates. Stack 1 contains two crates: crate Z is on the bottom, and crate N is on top. Stack 2 contains three crates; from bottom to top, they are crates M, C, and D. Finally, stack 3 contains a single crate, P.

Then, the rearrangement procedure is given. In each step of the procedure, a quantity of crates is moved from one stack to a different stack. In the first step of the above rearrangement procedure, one crate is moved from stack 2 to stack 1, resulting in this configuration:

```
[D]        
[N] [C]    
[Z] [M] [P]
 1   2   3
```

In the second step, three crates are moved from stack 1 to stack 3. Crates are moved one at a time, so the first crate to be moved (D) ends up below the second and third crates:

```
        [Z]
        [N]
    [C] [D]
    [M] [P]
 1   2   3
```

Then, both crates are moved from stack 2 to stack 1. Again, because crates are moved one at a time, crate C ends up below crate M:

```
        [Z]
        [N]
[M]     [D]
[C]     [P]
 1   2   3
```

Finally, one crate is moved from stack 1 to stack 2:

```

        [Z]
        [N]
        [D]
[C] [M] [P]
 1   2   3
```

The Elves just need to know which crate will end up on top of each stack; in this example, the top crates are C in stack 1, M in stack 2, and Z in stack 3, so you should combine these together and give the Elves the message CMZ.

After the rearrangement procedure completes, what crate ends up on top of each stack?

## Solution

First, let's just load in our data and see what we're working with:

In [1]:
# We will start by reading in the file, one line at a time, using the
# readlines() method. As a reminder, the
#
#   with open('filename', 'r') as f
#
# line is called a context manager in Python: in this case, it's an
# easy way to handle a lot of the nitty gritty of opening / reading /closing
# files.
with open('/content/lab-002_supply-stacks.txt', 'r') as f:
    data = [line.strip() for line in f.readlines()]

for line in data[:15]:
    print(line)

[N] [G]                     [Q]
[H] [B]         [B] [R]     [H]
[S] [N]     [Q] [M] [T]     [Z]
[J] [T]     [R] [V] [H]     [R] [S]
[F] [Q]     [W] [T] [V] [J] [V] [M]
[W] [P] [V] [S] [F] [B] [Q] [J] [H]
[T] [R] [Q] [B] [D] [D] [B] [N] [N]
[D] [H] [L] [N] [N] [M] [D] [D] [B]
1   2   3   4   5   6   7   8   9

move 3 from 1 to 2
move 1 from 7 to 1
move 1 from 6 to 5
move 5 from 5 to 9
move 2 from 5 to 2


Since this is all just raw text strings, let's start by splitting the list of strings on the first empty line that we see. Looking through the input file, we can note that each line is one of 3 cases:

  1. The line contains information about the crates
  2. The line contains information about the crate movement instructions
  3. The line is blank

  Given this, we know that we can just find the index position of the blank line and split our list into respective "crate stacks" and "instructions" lists

In [2]:
split_pos = data.index("")

# list slicing
# variable[start_position (inclusive) : end_position (exclusive)]
stacks = data[:split_pos]
instructions = data[split_pos + 1:]

In [3]:
stacks

['[N] [G]                     [Q]',
 '[H] [B]         [B] [R]     [H]',
 '[S] [N]     [Q] [M] [T]     [Z]',
 '[J] [T]     [R] [V] [H]     [R] [S]',
 '[F] [Q]     [W] [T] [V] [J] [V] [M]',
 '[W] [P] [V] [S] [F] [B] [Q] [J] [H]',
 '[T] [R] [Q] [B] [D] [D] [B] [N] [N]',
 '[D] [H] [L] [N] [N] [M] [D] [D] [B]',
 '1   2   3   4   5   6   7   8   9']

Now we need to figure out how many stacks of crates there are. Since any individual row of crates in our `stacks` list can have blanks, we need to look at the bottom-most row of data which contains the column number of each stack.

We'll use the `list[-1]` syntax to grab that last element of the list, and then split on whitespace so that we're just left with the number parts of the line. Lastly, we really only care about the final number in the sequence since that gives us the total number of stacks, so we'll again use that `list[-1]` syntax to just extract (in this case) the number `9`:

In [4]:
num_stacks = int(stacks[-1].split()[-1])
num_stacks

9

Since we know how many stacks of crates we now need, let's initialize an empty list of lists to represent each of the stacks in that range:

In [5]:
stacks_list = [[] for _ in range(num_stacks)]
stacks_list

[[], [], [], [], [], [], [], [], []]

Our basic data structure is looking solid, so now we need to parse the actual crates and figure out which crate belongs in which stack, and how high or low it is in the stack.

To do this, we can capitalize on the fact that our data are regular and have a repeating pattern. All rows of crates follow the same pattern:

  - `[`
  - Crate name (a capital letter)
  - `]`
  - ` ` (a blank space)

and this pattern repeats across the line. Since we know that every 4th character in each row can be a crate name, we can skip over every index that's not a multiple of 4 and just ignore that information as extraneous. To figure out whether there actually is or isn't a crate in each spot, we can use the `str.isalpha()` function to check whether the character looks like a letter in the range `[a-zA-Z]`:

In [13]:
for line_idx, line in enumerate(stacks[:-1][::-1]):
    print(f"Line {line_idx}:")
    print(f"\t{line}")
    print(f"\t{[
        (char, (char_idx // 4))
        for char_idx, char in enumerate(line)
        if char.isalpha()
    ]}")

Line 0:
	[D] [H] [L] [N] [N] [M] [D] [D] [B]
	[('D', 0), ('H', 1), ('L', 2), ('N', 3), ('N', 4), ('M', 5), ('D', 6), ('D', 7), ('B', 8)]
Line 1:
	[T] [R] [Q] [B] [D] [D] [B] [N] [N]
	[('T', 0), ('R', 1), ('Q', 2), ('B', 3), ('D', 4), ('D', 5), ('B', 6), ('N', 7), ('N', 8)]
Line 2:
	[W] [P] [V] [S] [F] [B] [Q] [J] [H]
	[('W', 0), ('P', 1), ('V', 2), ('S', 3), ('F', 4), ('B', 5), ('Q', 6), ('J', 7), ('H', 8)]
Line 3:
	[F] [Q]     [W] [T] [V] [J] [V] [M]
	[('F', 0), ('Q', 1), ('W', 3), ('T', 4), ('V', 5), ('J', 6), ('V', 7), ('M', 8)]
Line 4:
	[J] [T]     [R] [V] [H]     [R] [S]
	[('J', 0), ('T', 1), ('R', 3), ('V', 4), ('H', 5), ('R', 7), ('S', 8)]
Line 5:
	[S] [N]     [Q] [M] [T]     [Z]
	[('S', 0), ('N', 1), ('Q', 3), ('M', 4), ('T', 5), ('Z', 7)]
Line 6:
	[H] [B]         [B] [R]     [H]
	[('H', 0), ('B', 1), ('B', 4), ('R', 5), ('H', 7)]
Line 7:
	[N] [G]                     [Q]
	[('N', 0), ('G', 1), ('Q', 7)]


# Part Two

As you watch the crane operator expertly rearrange the crates, you notice the process isn't following your prediction.

Some mud was covering the writing on the side of the crane, and you quickly wipe it away. The crane isn't a CrateMover 9000 - it's a CrateMover 9001.

The CrateMover 9001 is notable for many new and exciting features: air conditioning, leather seats, an extra cup holder, and the ability to pick up and move multiple crates at once.

Again considering the example above, the crates begin in the same configuration:

```
    [D]    
[N] [C]    
[Z] [M] [P]
 1   2   3

move 1 from 2 to 1
move 3 from 1 to 3
move 2 from 2 to 1
move 1 from 1 to 2
```


Moving a single crate from stack 2 to stack 1 behaves the same as before:

```
[D]        
[N] [C]    
[Z] [M] [P]
 1   2   3
```

However, the action of moving three crates from stack 1 to stack 3 means that those three moved crates stay in the same order, resulting in this new configuration:

```
        [D]
        [N]
    [C] [Z]
    [M] [P]
 1   2   3
```

Next, as both crates are moved from stack 2 to stack 1, they retain their order as well:

```
        [D]
        [N]
[C]     [Z]
[M]     [P]
 1   2   3
```

Finally, a single crate is still moved from stack 1 to stack 2, but now it's crate C that gets moved:

```
        [D]
        [N]
        [Z]
[M] [C] [P]
 1   2   3
```

In this example, the CrateMover 9001 has put the crates in a totally different order: MCD.

Before the rearrangement process finishes, update your simulation so that the Elves know where they should stand to be ready to unload the final supplies. After the rearrangement procedure completes, what crate ends up on top of each stack?

In [ ]:
with open('lab_02.txt', 'r') as f:
    data = [line.strip() for line in f.readlines()]